# Lab 4: Docker Compose + Apache Kafka

## Cel

- Uruchomienie klastra Kafka w Docker Compose,
- Tworzenie tematów (topics),
- Wysyłanie i odbieranie wiadomości z poziomu terminala,
- Zrozumienie architektury: broker, temat, partycja, offset.

---

## Część 1: Uruchomienie Kafki

### Zadanie 1.1 — Docker Compose z Kafka

Utwórz plik `docker-compose.yml`, który uruchomi:

1. **Zookeeper** (zarządza klastrem Kafki)
2. **Kafka** (broker wiadomości)

Użyj obrazów `confluentinc/cp-zookeeper:7.6.0` i `confluentinc/cp-kafka:7.6.0`.

In [ ]:
%%file docker-compose.yml
services:
  zookeeper:
    image: confluentinc/cp-zookeeper:7.6.0
    environment:
      ZOOKEEPER_CLIENT_PORT: 2181

  kafka:
    image: confluentinc/cp-kafka:7.6.0
    depends_on:
      - zookeeper
    ports:
      - "9092:9092"
    environment:
      KAFKA_BROKER_ID: 1
      KAFKA_ZOOKEEPER_CONNECT: zookeeper:2181
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://localhost:9092
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 1

Uruchom:
```bash
docker compose up -d
docker compose ps
```

Poczekaj ok. 30 sekund, aż Kafka się uruchomi.

### Zadanie 1.2 — Utwórz temat

Utwórz temat `test-topic` z 3 partycjami:

In [ ]:
# W terminalu:
# docker compose exec kafka kafka-topics --create \
#   --topic test-topic \
#   --bootstrap-server localhost:9092 \
#   --partitions 3 \
#   --replication-factor 1

# Sprawdz liste tematow:
# docker compose exec kafka kafka-topics --list --bootstrap-server localhost:9092

# Wklej wynik:
# TWÓJ WYNIK

### Zadanie 1.3 — Wyślij i odbierz wiadomości z terminala

Otwórz **dwa terminale**.

Terminal 1 — konsument (najpierw uruchom!):
```bash
docker compose exec kafka kafka-console-consumer \
  --topic test-topic \
  --bootstrap-server localhost:9092 \
  --from-beginning
```

Terminal 2 — producent:
```bash
docker compose exec kafka kafka-console-producer \
  --topic test-topic \
  --bootstrap-server localhost:9092
```

Wpisz kilka wiadomości w terminalu producenta i obserwuj, jak pojawiają się u konsumenta.

### Zadanie 1.4 — Szczegóły tematu

Sprawdź szczegóły tematu (partycje, lider, repliki):

In [ ]:
# docker compose exec kafka kafka-topics --describe \
#   --topic test-topic \
#   --bootstrap-server localhost:9092

# Wklej wynik i odpowiedz:
# 1. Ile partycji ma temat?
# 2. Który broker jest liderem?
# TWÓJ WYNIK

---

## Część 2: Kafka z Pythona

Zainstaluj bibliotekę:
```bash
pip install confluent-kafka
```

### Zadanie 2.1 — Producent w Pythonie

Napisz producenta, który wysyła 20 wiadomości JSON do tematu `transakcje`.
Każda wiadomość powinna zawierać: `id`, `kwota`, `sklep`, `timestamp`.

In [ ]:
%%file producer.py
from confluent_kafka import Producer
import json, random, time
from datetime import datetime

conf = {'bootstrap.servers': 'localhost:9092'}
producer = Producer(conf)

def delivery_report(err, msg):
    if err:
        print(f'Błąd: {err}')
    else:
        print(f'Wysłano do {msg.topic()} [{msg.partition()}] offset={msg.offset()}')

# TWÓJ KOD
# Petla wysylajaca 20 transakcji (po jednej na sekunde)
# Kazda transakcja to JSON z polami: id, kwota, sklep, timestamp
# Uzyj producer.produce(topic, value=json.dumps(msg).encode('utf-8'), callback=delivery_report)
# Na koncu: producer.flush()

### Zadanie 2.2 — Konsument w Pythonie

Napisz konsumenta, który:
1. Subskrybuje temat `transakcje`
2. Odczytuje wiadomości
3. Wyświetla ALERT dla transakcji powyżej 5000 PLN

In [ ]:
%%file consumer.py
from confluent_kafka import Consumer
import json

conf = {
    'bootstrap.servers': 'localhost:9092',
    'group.id': 'fraud-detector',
    'auto.offset.reset': 'earliest'
}

consumer = Consumer(conf)
# TWÓJ KOD
# Subskrybuj temat 'transakcje'
# W petli: pobieraj wiadomosci (consumer.poll(1.0))
# Dekoduj JSON i wyswietl ALERT jesli kwota > 5000

### Zadanie 2.3 — Przetestuj

Uruchom w dwóch terminalach:
```bash
# Terminal 1:
python consumer.py

# Terminal 2:
python producer.py
```

Obserwuj, jak konsument reaguje na wiadomości producenta.

---

## Część 3: Wiele konsumentów

### Zadanie 3.1 — Grupy konsumentów

Uruchom **dwa** konsumenty z tą samą `group.id` i jednego producenta.

1. Jak Kafka rozdziela wiadomości między konsumentów?
2. Co się stanie, jeśli wyłączysz jednego konsumenta?

In [ ]:
# Odpowiedz na pytania:
# 1. ...
# 2. ...

---

## Praca domowa

1. Utwórz temat `zamowienia` z 5 partycjami.
2. Napisz producenta, który wysyła zamówienia z kluczem = `miasto` (użyj parametru `key` w `produce()`).
3. Sprawdź, że wiadomości z tego samego miasta trafiają do tej samej partycji.
4. Wyślij kod do repozytorium Git.

Na następnych zajęciach: zaawansowany producent/konsument Kafka w Pythonie.